In [1]:
import numpy as np
import json
from PIL import Image
import cv2
from pathlib import Path
from collections import defaultdict
from multiprocessing import Pool, cpu_count


def create_class_regions(semantic_mask):
    class_regions = {}
    unique_classes = np.unique(semantic_mask)
    for class_id in unique_classes:
        region_mask = (semantic_mask == class_id).astype(np.uint8)
        class_regions[int(class_id)] = region_mask
        print(f"Class {class_id}: Region area = {np.sum(region_mask)} pixels")
    return class_regions


def assign_objects_to_classes(object_mask, class_regions):
    object_to_class = {}
    unique_objects = np.unique(object_mask)
    unique_objects = unique_objects[unique_objects != 0]
    MIN_OVERLAP_THRESHOLD = 30

    for obj_id in unique_objects:
        obj_pixels = (object_mask == obj_id)
        obj_area = np.sum(obj_pixels)
        best_class = None
        best_overlap = 0
        best_overlap_pct = 0

        for class_id, region_mask in class_regions.items():
            if class_id == -1:
                continue
            overlap = np.sum(obj_pixels & (region_mask > 0))
            overlap_pct = (overlap / obj_area * 100) if obj_area > 0 else 0
            if overlap > best_overlap:
                best_overlap = overlap
                best_overlap_pct = overlap_pct
                best_class = class_id

        if best_overlap_pct >= MIN_OVERLAP_THRESHOLD:
            object_to_class[int(obj_id)] = best_class
        else:
            object_to_class[int(obj_id)] = None

    return object_to_class


def save_object_overlay(object_mask, object_to_class, tiff_path, output_dir):
    class_colors = {
        0: [128, 128, 128],  # Gray - Background
        1: [0, 0, 255],      # Blue - Cortex
        2: [0, 255, 0],      # Green - Phloem Fibers
        3: [128, 0, 128],    # Purple - Phloem
        4: [255, 0, 0],      # Red - Hydrated Xylem vessels
        5: [255, 255, 0],    # Yellow - Air-based Pith cells
        6: [255, 165, 0],    # Orange - Water-based Pith cells
        7: [255, 102, 178],  # Pink - Dehydrated Xylem vessels
    }

    original_img = np.array(Image.open(tiff_path))
    if len(original_img.shape) == 2:
        original_img = np.stack([original_img] * 3, axis=-1)
    if original_img.max() <= 1.0:
        original_img = (original_img * 255).astype(np.uint8)
    original_img = original_img.astype(np.uint8)

    colored_objects = np.zeros_like(original_img, dtype=np.uint8)
    for obj_id, class_id in object_to_class.items():
        if class_id is not None:
            obj_pixels = (object_mask == obj_id)
            colored_objects[obj_pixels] = class_colors.get(class_id, [128, 128, 128])

    alpha = 0.8
    overlay = cv2.addWeighted(original_img, 1 - alpha, colored_objects, alpha, 0)

    out_path = Path(output_dir) / f"{Path(tiff_path).stem}_overlay.png"
    cv2.imwrite(str(out_path), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    print(f"Overlay saved to: {out_path}")


def process_single_file(tiff_path, mask_tiff_path, semantic_mask_path, mappings_dir, overlay_dir):
    print(f"\n{'='*80}")
    print(f"Processing: {Path(tiff_path).name}")
    print(f"{'='*80}")

    object_mask = np.array(Image.open(mask_tiff_path))
    print(f"Object mask shape: {object_mask.shape}")
    print(f"Number of unique objects: {len(np.unique(object_mask)) - 1}")

    semantic_mask = np.load(semantic_mask_path)
    print(f"Semantic mask shape: {semantic_mask.shape}")

    class_regions = create_class_regions(semantic_mask)
    print(f"\nClasses found: {sorted(class_regions.keys())}")

    object_to_class = assign_objects_to_classes(object_mask, class_regions)

    class_names = {
        0: 'Background', 1: 'Cortex', 2: 'Phloem Fibers',
        3: 'Phloem', 4: 'Hydrated Xylem vessels', 5: 'Air-based Pith cells',
        6: 'Water-based Pith cells', 7: 'Dehydrated Xylem vessels'
    }
    class_counts = defaultdict(int)
    unassigned_count = 0
    for obj_id, class_id in object_to_class.items():
        if class_id is not None:
            class_counts[class_id] += 1
        else:
            unassigned_count += 1

    print("\nSummary:")
    for class_id in sorted(class_regions.keys()):
        print(f"Class {class_id} ({class_names.get(class_id, 'Unknown')}): {class_counts.get(class_id, 0)} objects")
    print(f"Unassigned: {unassigned_count} objects")

    save_object_overlay(object_mask, object_to_class, tiff_path, overlay_dir)

    base_name = Path(tiff_path).stem
    out_path = Path(mappings_dir) / f"{base_name}_mapping.json"
    with open(out_path, 'w') as f:
        json.dump(object_to_class, f, indent=2)
    print(f"Mapping saved to: {out_path}")

    return object_to_class, class_regions


def _process_worker(args):
    """Top-level wrapper so the function is picklable for multiprocessing."""
    image_path, mask_path, semantic_mask_path, mappings_dir, overlay_dir = args
    try:
        process_single_file(image_path, mask_path, semantic_mask_path, mappings_dir, overlay_dir)
        return (str(image_path), True, None)
    except Exception as e:
        import traceback
        return (str(image_path), False, traceback.format_exc())


def batch_process(images_dir, masks_dir, annotations_dir, mappings_dir, overlay_dir):
    mappings_dir = Path(mappings_dir)
    mappings_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir = Path(overlay_dir)
    overlay_dir.mkdir(parents=True, exist_ok=True)

    images_dir = Path(images_dir)
    image_files = list(images_dir.glob("*.tiff")) + list(images_dir.glob("*.tif"))
    print(f"Found {len(image_files)} image files")

    tasks = []
    skipped_count = 0

    for image_path in sorted(image_files):
        base_name = image_path.stem
        masks_dir_path = Path(masks_dir)
        mask_path = masks_dir_path / f"{base_name}_masks.tif"
        if not mask_path.exists():
            mask_path = masks_dir_path / f"{base_name}_masks.tiff"

        semantic_mask_path = Path(annotations_dir) / f"{base_name}_mask.npy"

        if not mask_path.exists():
            print(f"\nSkipping {base_name}: mask file not found")
            skipped_count += 1
            continue
        if not semantic_mask_path.exists():
            print(f"\nSkipping {base_name}: semantic mask file not found")
            skipped_count += 1
            continue

        tasks.append((image_path, mask_path, semantic_mask_path, mappings_dir, overlay_dir))

    num_workers = min(cpu_count(), len(tasks)) if tasks else 1
    print(f"Using {num_workers} worker(s) for {len(tasks)} file(s)")

    processed_count = 0
    with Pool(processes=num_workers) as pool:
        for image_path, success, tb in pool.imap_unordered(_process_worker, tasks):
            if success:
                processed_count += 1
            else:
                print(f"\nError processing {image_path}:\n{tb}")
                skipped_count += 1

    print(f"\n{'='*80}")
    print(f"Batch processing complete! Processed: {processed_count} | Skipped: {skipped_count}")
    print(f"{'='*80}")


if __name__ == "__main__":
    images_dir      = "./data/images"
    masks_dir       = "./data/instance_masks"
    annotations_dir = "./data/annotations"
    mappings_dir    = "./data/mappings"
    overlay_dir     = "./data/assign_id"

    batch_process(images_dir, masks_dir, annotations_dir, mappings_dir, overlay_dir)

Found 42 image files
Using 42 worker(s) for 42 file(s)




Processing: 20211222_094342_petiole_test_00100.tiff=============================



Processing: 20211222_094342_petiole_test_00200.tiff
Processing: 20211222_094342_petiole_test_00400.tiff=============================
Processing: 20211222_094342_petiole_test_00300.tiff=============================



Processing: 20211222_094342_petiole_test_00500.tiff



Processing: 20211222_104840_petiole_test2_00200.tiff
Processing: 20211222_104840_petiole_test2_00100.tiff============================




Processing: 20211222_104840_petiole_test2_00400.tiff============================


Processing: 20211222_104840_petiole_test2_00300.tiffProcessing: 20211222_104840_petiole_test2_00500.tiff


Processing: 20211222_113313_petiole3_00200.tiff=================================
Processing: 20211222_113313_petiole3_00300.tiff=================================
Processing: 20211222_113313_petiole3_00400.tiff=================================





Processin

In [2]:
# change class id = (null and -1 and 255) to 0 (background), which is ignored in coco

In [3]:
import json
from pathlib import Path
from collections import defaultdict

def process_mapping_file(mapping_path):
    """
    Process a single mapping file: change null or -1 or 255 classes to 0 and save as _corrected.json
    
    Args:
        mapping_path: path to the mapping JSON file
    """
    base_name = mapping_path.stem
    
    print(f"\n{'='*80}")
    print(f"Processing: {mapping_path.name}")
    print(f"{'='*80}")
    
    # Load the object class mapping
    with open(mapping_path, 'r') as f:
        object_to_class = json.load(f)
    
    print(f"Total objects in mapping: {len(object_to_class)}")
    
    # Count objects before update
    null_count = 0
    negative_one_count = 0
    two_fifty_five_count = 0
    for obj_id, class_id in object_to_class.items():
        if class_id is None or class_id == 'null':
            null_count += 1
        elif class_id == -1 or class_id == '-1':
            negative_one_count += 1
        elif class_id == 255 or class_id == '255':
            two_fifty_five_count += 1
    
    print(f"Objects with null class: {null_count}")
    print(f"Objects with -1 class: {negative_one_count}")
    print(f"Objects with 255 class: {two_fifty_five_count}")
    
    # Update the mapping - change all null, -1, and 255 to class 0
    updated_count = 0
    for obj_id, class_id in object_to_class.items():
        if class_id is None or class_id == 'null' or class_id == -1 or class_id == '-1' or class_id == 255 or class_id == '255':
            object_to_class[obj_id] = 0
            updated_count += 1
    
    print(f"Updated {updated_count} objects from null/-1/255 to class 0")
    
    # Save the updated mapping to a new file with _corrected suffix
    output_path = mapping_path.parent / f"{base_name}_corrected{mapping_path.suffix}"
    
    with open(output_path, 'w') as f:
        json.dump(object_to_class, f, indent=2)
    
    print(f"\n✓ Updated mapping saved to: {output_path}")
    
    # Print summary statistics
    class_counts = defaultdict(int)
    unassigned_count = 0
    
    for obj_id, class_id in object_to_class.items():
        if class_id is not None and class_id != 'null' and class_id != -1 and class_id != '-1' and class_id != 255 and class_id != '255':
            class_counts[class_id] += 1
        else:
            unassigned_count += 1
    
    print("\nUpdated class distribution:")
    print("-" * 50)
    for class_id in sorted(class_counts.keys(), key=str):
        count = class_counts[class_id]
        print(f"Class {class_id}: {count} objects")
    if unassigned_count > 0:
        print(f"Unassigned: {unassigned_count} objects")
    
    return updated_count


def batch_process_mappings(mappings_dir):
    """
    Process all mapping JSON files in the specified directory.
    
    Args:
        mappings_dir: directory containing mapping JSON files
    """
    mappings_dir = Path(mappings_dir)
    
    # Get all JSON files that don't already end with _corrected.json
    mapping_files = [f for f in mappings_dir.glob("*.json") 
                     if not f.stem.endswith("_corrected")]
    
    print(f"Found {len(mapping_files)} mapping files to process")
    
    processed_count = 0
    total_updated_objects = 0
    
    for mapping_path in sorted(mapping_files):
        try:
            updated_count = process_mapping_file(mapping_path)
            total_updated_objects += updated_count
            processed_count += 1
        except Exception as e:
            print(f"\nError processing {mapping_path.name}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*80}")
    print("Batch processing complete!")
    print(f"Processed: {processed_count} files")
    print(f"Total objects updated from null/-1/255 to class 0: {total_updated_objects}")
    print(f"{'='*80}")


if __name__ == "__main__":
    # Process all mapping files in the folder
    mappings_dir = "./data/mappings/"
    
    batch_process_mappings(mappings_dir)

Found 42 mapping files to process

Processing: 20211222_094342_petiole_test_00100_mapping.json
Total objects in mapping: 1654
Objects with null class: 0
Objects with -1 class: 0
Objects with 255 class: 30
Updated 30 objects from null/-1/255 to class 0

✓ Updated mapping saved to: data/mappings/20211222_094342_petiole_test_00100_mapping_corrected.json

Updated class distribution:
--------------------------------------------------
Class 0: 44 objects
Class 1: 1326 objects
Class 2: 28 objects
Class 3: 1 objects
Class 4: 51 objects
Class 5: 17 objects
Class 6: 107 objects
Class 7: 80 objects

Processing: 20211222_094342_petiole_test_00200_mapping.json
Total objects in mapping: 1790
Objects with null class: 0
Objects with -1 class: 0
Objects with 255 class: 22
Updated 22 objects from null/-1/255 to class 0

✓ Updated mapping saved to: data/mappings/20211222_094342_petiole_test_00200_mapping_corrected.json

Updated class distribution:
--------------------------------------------------
Class 

In [4]:
# split mapping into two files, save to meta folder

In [5]:
import numpy as np
import json
from PIL import Image
from pathlib import Path
from collections import defaultdict
import cv2

def process_single_file(image_path, mask_path, mapping_path, output_dir):
    """
    Process a single set of files and generate two .txt files.
    
    Args:
        image_path: path to the original TIFF image
        mask_path: path to the object mask TIFF
        mapping_path: path to the corrected mapping JSON file
        output_dir: directory to save the .txt files
    """
    base_name = Path(image_path).stem
    
    print(f"\n{'='*80}")
    print(f"Processing: {base_name}")
    print(f"{'='*80}")
    
    print("Loading files...")
    original_img = np.array(Image.open(image_path))
    object_mask = np.array(Image.open(mask_path))
    
    with open(mapping_path, 'r') as f:
        object_to_class = json.load(f)
    
    print(f"Image shape: {original_img.shape}")
    print(f"Mask shape: {object_mask.shape}")
    print(f"Total objects in mapping: {len(object_to_class)}")
    
    # Count objects by category
    class_counts = defaultdict(int)
    category_names = {
        '0': 'Background',
        '1': 'Cortex',
        '2': 'Phloem Fibers',
        '3': 'Phloem',
        '4': 'Hydrated Xylem vessels',
        '5': 'Air-based Pith cells',
        '6': 'Water-based Pith cells',
        '7': 'Dehydrated Xylem vessels'
    }
    
    for obj_id, class_id in object_to_class.items():
        if class_id is not None and class_id != 'null':
            class_counts[class_id] += 1
    
    total_annotated = sum(class_counts.values())
    
    print(f"\nTotal annotated objects: {total_annotated}")
    # FIX: Convert to string for sorting
    for class_id in sorted(class_counts.keys(), key=str):
        count = class_counts[class_id]
        name = category_names.get(str(class_id), f'Class {class_id}')
        print(f"Category {class_id} ({name}): {count} objects")
    
    # Get all unique objects from mask
    unique_objects = np.unique(object_mask)
    unique_objects = unique_objects[unique_objects != 0]  # Remove background
    total_objects = len(unique_objects)
    
    print(f"\nTotal objects in mask: {total_objects}")
    
    # ========== File 1: object_class_mapping.txt ==========
    output_path1 = output_dir / f"{base_name}_object_class_mapping.txt"
    with open(output_path1, 'w') as f:
        f.write("ObjectID\tClassID\n")
        
        # Write only annotated objects (those with class assignments)
        # FIX: Handle both string and int object IDs safely
        sorted_items = sorted(
            object_to_class.items(), 
            key=lambda x: int(x[0]) if str(x[0]).isdigit() else x[0]
        )
        
        for obj_id, class_id in sorted_items:
            if class_id is not None and class_id != 'null':
                f.write(f"{obj_id}\t{class_id}\n")
    
    print(f"\n✓ Saved File 1: {output_path1}")
    print(f"  Format: ObjectID, ClassID")
    print(f"  Total objects: {total_annotated}")
    
    # ========== File 2: object_geometry.txt ==========
    print("\nExtracting object geometries...")
    output_path2 = output_dir / f"{base_name}_object_geometry.txt"
    
    with open(output_path2, 'w') as f:
        f.write("ObjectID\tPolygon\tBBox\n")
        
        for obj_id in unique_objects:
            # Create binary mask for this object
            obj_pixels = (object_mask == obj_id).astype(np.uint8)
            
            # Find contours
            contours, _ = cv2.findContours(obj_pixels, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if len(contours) > 0:
                # Get the largest contour
                contour = max(contours, key=cv2.contourArea)
                
                # Extract polygon coordinates
                polygon_points = contour.squeeze()
                
                # Handle single point or line cases
                if polygon_points.ndim == 1:
                    polygon_points = polygon_points.reshape(1, -1)
                
                # Format polygon as "x1,y1;x2,y2;..."
                polygon_str = ";".join([f"{pt[0]},{pt[1]}" for pt in polygon_points])
                
                # Get bounding box in x1,y1,x2,y2 format
                x, y, w, h = cv2.boundingRect(contour)
                x1, y1 = x, y
                x2, y2 = x + w, y + h
                bbox_str = f"{x1},{y1},{x2},{y2}"
                
                # Write to file with tab separation
                f.write(f"{obj_id}\t{polygon_str}\t{bbox_str}\n")
    
    print(f"\n✓ Saved File 2: {output_path2}")
    print(f"  Format: ObjectID, Polygon, BBox")
    print(f"  Total objects: {total_objects}")
    
    return total_annotated, total_objects


def batch_process(images_dir, masks_dir, mappings_dir, meta_dir):
    """
    Process all matching files in the specified directories.
    
    Args:
        images_dir: directory containing image files (.tiff or .tif)
        masks_dir: directory containing mask files (*_masks.tif or *_masks.tiff)
        mappings_dir: directory containing corrected mapping files (*_corrected.json)
        meta_dir: directory to save the .txt metadata files
    """
    # Create output directory if it doesn't exist
    meta_dir = Path(meta_dir)
    meta_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all image files
    images_dir = Path(images_dir)
    image_files = list(images_dir.glob("*.tiff")) + list(images_dir.glob("*.tif"))
    
    print(f"Found {len(image_files)} image files")
    
    processed_count = 0
    skipped_count = 0
    
    for image_path in sorted(image_files):
        base_name = image_path.stem
        
        # Find corresponding mask file
        masks_dir_path = Path(masks_dir)
        mask_path = masks_dir_path / f"{base_name}_masks.tif"
        if not mask_path.exists():
            mask_path = masks_dir_path / f"{base_name}_masks.tiff"
        
        # Find corresponding corrected mapping file
        mappings_dir_path = Path(mappings_dir)
        mapping_path = mappings_dir_path / f"{base_name}_mapping_corrected.json"
        
        # Check if all files exist
        if not mask_path.exists():
            print(f"\nSkipping {base_name}: mask file not found")
            skipped_count += 1
            continue
        
        if not mapping_path.exists():
            print(f"\nSkipping {base_name}: corrected mapping file not found")
            skipped_count += 1
            continue
        
        # Process the files
        try:
            process_single_file(image_path, mask_path, mapping_path, meta_dir)
            processed_count += 1
        except Exception as e:
            print(f"\nError processing {base_name}: {str(e)}")
            import traceback
            traceback.print_exc()
            skipped_count += 1
    
    print(f"\n{'='*80}")
    print("Batch processing complete!")
    print(f"Processed: {processed_count} files")
    print(f"Skipped: {skipped_count} files")
    print(f"Output directory: {meta_dir}")
    print(f"{'='*80}")


if __name__ == "__main__":
    # Batch process all files
    images_dir = "./data/images"
    masks_dir = "./data/instance_masks"
    mappings_dir = "./data/mappings"
    meta_dir = "./data/meta"
    
    batch_process(images_dir, masks_dir, mappings_dir, meta_dir)

Found 42 image files

Processing: 20211222_094342_petiole_test_00100
Loading files...
Image shape: (2560, 2560)
Mask shape: (2560, 2560)
Total objects in mapping: 1654

Total annotated objects: 1654
Category 0 (Background): 44 objects
Category 1 (Cortex): 1326 objects
Category 2 (Phloem Fibers): 28 objects
Category 3 (Phloem): 1 objects
Category 4 (Hydrated Xylem vessels): 51 objects
Category 5 (Air-based Pith cells): 17 objects
Category 6 (Water-based Pith cells): 107 objects
Category 7 (Dehydrated Xylem vessels): 80 objects

Total objects in mask: 1654

✓ Saved File 1: data/meta/20211222_094342_petiole_test_00100_object_class_mapping.txt
  Format: ObjectID, ClassID
  Total objects: 1654

Extracting object geometries...

✓ Saved File 2: data/meta/20211222_094342_petiole_test_00100_object_geometry.txt
  Format: ObjectID, Polygon, BBox
  Total objects: 1654

Processing: 20211222_094342_petiole_test_00200
Loading files...
Image shape: (2560, 2560)
Mask shape: (2560, 2560)
Total objects i

In [6]:
# remove class =0 (background) in meta folder

In [7]:
from pathlib import Path

def process_file_pair(mapping_file, geometry_file):
    """
    Process a pair of mapping and geometry files to remove class 0 objects.
    
    Args:
        mapping_file: path to object_class_mapping.txt
        geometry_file: path to object_geometry.txt
    """
    base_name = mapping_file.stem.replace("_object_class_mapping", "")
    
    print(f"\n{'='*80}")
    print(f"Processing: {base_name}")
    print(f"{'='*80}")
    
    # ========== Clean object_class_mapping.txt ==========
    print("Cleaning object_class_mapping.txt...")
    
    # First pass: identify objects with class 0
    class_0_objects = set()
    with open(mapping_file, 'r') as f:
        header = f.readline()  # Skip header
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                obj_id = parts[0]
                class_id = parts[1]
                if class_id == '0':
                    class_0_objects.add(obj_id)
    
    print(f"Found {len(class_0_objects)} objects with class ID 0")
    
    # Second pass: write cleaned file
    cleaned_lines = []
    removed_count = 0
    
    with open(mapping_file, 'r') as f:
        header = f.readline()
        cleaned_lines.append(header)
        
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                class_id = parts[1]
                if class_id != '0':
                    cleaned_lines.append(line)
                else:
                    removed_count += 1
    
    output_mapping = mapping_file.parent / f"{base_name}_object_class_mapping_cleaned.txt"
    with open(output_mapping, 'w') as f:
        f.writelines(cleaned_lines)
    
    print(f"✓ Removed {removed_count} objects from mapping file")
    print(f"✓ Saved to: {output_mapping}")
    
    # ========== Clean object_geometry.txt ==========
    print("\nCleaning object_geometry.txt...")
    cleaned_lines = []
    removed_count = 0
    
    with open(geometry_file, 'r') as f:
        header = f.readline()
        cleaned_lines.append(header)
        
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 1:
                obj_id = parts[0]
                if obj_id not in class_0_objects:
                    cleaned_lines.append(line)
                else:
                    removed_count += 1
    
    output_geometry = geometry_file.parent / f"{base_name}_object_geometry_cleaned.txt"
    with open(output_geometry, 'w') as f:
        f.writelines(cleaned_lines)
    
    print(f"✓ Removed {removed_count} objects from geometry file")
    print(f"✓ Saved to: {output_geometry}")
    
    return len(class_0_objects)


def batch_clean_meta(meta_dir):
    """
    Process all txt file pairs in the meta directory to remove class 0 objects.
    
    Args:
        meta_dir: directory containing the txt metadata files
    """
    meta_dir = Path(meta_dir)
    
    # Find all object_class_mapping.txt files (excluding _cleaned versions)
    mapping_files = [f for f in meta_dir.glob("*_object_class_mapping.txt")
                     if not f.stem.endswith("_cleaned")]
    
    print(f"Found {len(mapping_files)} mapping files to process")
    
    processed_count = 0
    total_removed_objects = 0
    
    for mapping_file in sorted(mapping_files):
        # Find corresponding geometry file
        base_name = mapping_file.stem.replace("_object_class_mapping", "")
        geometry_file = meta_dir / f"{base_name}_object_geometry.txt"
        
        if not geometry_file.exists():
            print(f"\nSkipping {base_name}: geometry file not found")
            continue
        
        try:
            removed_count = process_file_pair(mapping_file, geometry_file)
            total_removed_objects += removed_count
            processed_count += 1
        except Exception as e:
            print(f"\nError processing {base_name}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*80}")
    print("Batch cleaning complete!")
    print(f"Processed: {processed_count} file pairs")
    print(f"Total objects with class 0 removed: {total_removed_objects}")
    print(f"{'='*80}")


if __name__ == "__main__":
    # Process all txt files in the meta folder
    meta_dir = "./data/meta"
    
    batch_clean_meta(meta_dir)

Found 42 mapping files to process

Processing: 20211222_094342_petiole_test_00100
Cleaning object_class_mapping.txt...
Found 44 objects with class ID 0
✓ Removed 44 objects from mapping file
✓ Saved to: data/meta/20211222_094342_petiole_test_00100_object_class_mapping_cleaned.txt

Cleaning object_geometry.txt...
✓ Removed 44 objects from geometry file
✓ Saved to: data/meta/20211222_094342_petiole_test_00100_object_geometry_cleaned.txt

Processing: 20211222_094342_petiole_test_00200
Cleaning object_class_mapping.txt...
Found 34 objects with class ID 0
✓ Removed 34 objects from mapping file
✓ Saved to: data/meta/20211222_094342_petiole_test_00200_object_class_mapping_cleaned.txt

Cleaning object_geometry.txt...
✓ Removed 34 objects from geometry file
✓ Saved to: data/meta/20211222_094342_petiole_test_00200_object_geometry_cleaned.txt

Processing: 20211222_094342_petiole_test_00300
Cleaning object_class_mapping.txt...
Found 54 objects with class ID 0
✓ Removed 54 objects from mapping file